# External Connection Configuration

> **Purpose:** Configure Unity Catalog connection to external storage (Azure Data Lake Storage Gen2) using Access Connector and External Location.

**Prerequisites:**
- Azure Databricks Workspace with Unity Catalog
- Azure Data Lake Storage Gen2 (ADLS)
- Administrator permissions in Azure Portal and Databricks Workspace

### Contents

| Step | Description |
|------|-------------|
| **Part 1** | Get Access Connector Resource ID |
| **Part 2** | Grant RBAC permissions on Storage Account |
| **Part 3** | Create Storage Credential in Databricks |
| **Part 4** | Create External Location |
| **Part 5** | Configure access permissions |

## Part 1: Get Access Connector Resource ID

Go to **Azure Portal** → your Workspace → **Managed Resource Group**.

![Managed Resource Group](../../assets/images/training_2026/cheatsheet/17cf776d15ce4ffcb87d64d7b28d6107.webp)

Inside the Managed Resource Group, find the **Access Connector for Azure Databricks** resource (`unity-access-connector`). Copy its **Resource ID** — you will need it in the following steps.

![Access Connector Resource ID](../../assets/images/training_2026/cheatsheet/952e125cc24944499947d4e95d089cc4.webp)

> **Note:** The Resource ID has the format:
> `/subscriptions/<sub-id>/resourceGroups/<rg>/providers/Microsoft.Databricks/accessConnectors/<name>`

## Part 2: Grant RBAC Permissions on Storage Account

Go to your **Azure Data Lake Storage Gen2** → **Access Control (IAM)**.

![Storage Account IAM](../../assets/images/training_2026/cheatsheet/a67bf2bdb6cb4a2cb6b30897a7182b2b.webp)

Add a role using the **Principle of Least Privilege**:

| Role | Permissions | Use Case |
|------|-------------|----------|
| **Storage Blob Data Reader** | Read only | Production (read-only) |
| **Storage Blob Data Contributor** | Read + Write | Training / development |

> **For training purposes**, assign the **Storage Blob Data Contributor** role.
> In production, always apply the minimum required scope of permissions.

![Role Assignment](../../assets/images/training_2026/cheatsheet/fdcf7b03b80e44c499a0f88147a69361.webp)

Go to the **Members** tab → **Select members** → choose type **Managed Identity** → filter by **Access Connector for Azure Databricks** and select your connector.

![Member Selection - Managed Identity](../../assets/images/training_2026/cheatsheet/36092c21a0034e12a87e2b5e2f624501.webp)

Click **Review + Assign** — done. The Access Connector now has permissions on the Storage Account.

## Part 3: Create Storage Credential in Databricks

Go to **Databricks Workspace** → **Catalog** → **External Data** → **Credentials**.

![Catalog - External Data](../../assets/images/training_2026/cheatsheet/d369852738144865a154b4e46bebe71a.webp)

Create a new credential (if one does not already exist):

| Field | Value |
|-------|-------|
| **Credential name** | Your chosen name (e.g., `adls_training_credential`) |
| **Access connector ID** | Resource ID copied in Part 1 |

> **Tip:** A Storage Credential is shared — it can be reused across multiple External Locations.

## Part 4: Create External Location

Go to the **External Locations** tab → click **Create External Location**.

Fill in the form:

| Field | Value | Example |
|-------|-------|---------|
| **External location name** | Custom name | `training_landing_zone` |
| **URL** | ABFSS path to container | `abfss://container@account.dfs.core.windows.net/path` |
| **Storage credential** | Credential from Part 3 | `adls_training_credential` |

> **Important:** Before creating the External Location, create a dedicated container in ADLS for the solution.

![External Location form](../../assets/images/training_2026/cheatsheet/e037cfc8cc554ee491f9367f3101b49b.webp)

Click **Create** and wait for connection validation. If you see the following output — configuration completed successfully:

![External Location success](../../assets/images/training_2026/cheatsheet/a1373fa1fe0741699a3d29fd38868640.webp)

## Part 5: Configure Access Permissions

Final step — in the **Permissions** tab of the External Location, grant appropriate permissions to users/groups.

| Permission | Description |
|------------|-------------|
| `READ FILES` | Read files from the location |
| `WRITE FILES` | Write files to the location |
| `CREATE EXTERNAL TABLE` | Create external tables |
| `ALL PRIVILEGES` | Full access (dev/demo only) |

> **WARNING:** For training purposes, `ALL PRIVILEGES` has been granted.
> **Never do this in production!** Apply granular permissions according to your organization's governance policy.

![Permissions](../../assets/images/training_2026/cheatsheet/419e8741a5304838931458986d191083.webp)

### Summary

```
Azure Portal                          Databricks Workspace
─────────────                         ────────────────────
1. Managed Resource Group             3. Catalog → Credentials
   └─ Access Connector (Resource ID)     └─ Storage Credential (name + Resource ID)

2. Storage Account → IAM              4. Catalog → External Locations
   └─ Role Assignment                    └─ External Location (URL + Credential)
     (Managed Identity + RBAC Role)

                                      5. External Location → Permissions
                                         └─ Granular permissions
```